In [1]:
!pip install shap lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=16c826e034cccfd17a857e29daeb978028b781a7cb71f97aac82eade6faede00
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [39]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression as SklearnLinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
from xgboost.spark import SparkXGBRegressor

import shap
import lime
import lime.lime_tabular

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor as SparkRFRegressor, LinearRegression as SparkLinearRegression, GBTRegressor as SparkGBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [40]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [41]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

MOVIES_CSV_PATH = "movies.csv"
BOX_OFFICE_CSV_PATH = "enhanced_box_office_data(2000-2024)u.csv"
TMDB_ALL_MOVIES_CSV_PATH = '/content/drive/MyDrive/Thesis_Dataset/TMDB_all_movies.csv'

# chronological split: train <= 2019, test 2020-2026
TRAIN_MAX_YEAR = 2019
MIN_YEAR, MAX_YEAR = 2000, 2026

GENRE_LIST = ["Drama", "Comedy", "Thriller", "Action", "Romance", "Horror", "Adventure", "Crime", "Science Fiction", "Fantasy", "Family", "Animation"]

os.makedirs("visualizations", exist_ok=True)

## Spark Session Setup and Datasets Loading

In [42]:
print("Initializing Spark Session...")
spark = SparkSession.builder \
    .appName("MovieBoxOfficeForecasting") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")


def normalize_title_col(col):
    lowered = F.lower(F.trim(col))
    no_punct = F.regexp_replace(lowered, r"[^a-z0-9]+", " ")
    return F.trim(F.regexp_replace(no_punct, r"\\s+", " "))

Initializing Spark Session...


In [43]:
# dataset 1: movies.csv
movies_raw = (
    spark.read
    .option("header", True)
    .option("quote", "\"")
    .option("escape", "\"")
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .csv(MOVIES_CSV_PATH)
)

movies = (
    movies_raw
    .withColumn("budget", F.col("budget").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("runtime", F.col("runtime").cast("double"))
    .withColumn("vote_average", F.col("vote_average").cast("double"))
    .withColumn("vote_count", F.col("vote_count").cast("double"))
    .withColumn("popularity", F.col("popularity").cast("double"))
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("release_year", F.year("release_date"))
    .withColumn("release_month", F.month("release_date"))
    .withColumn("norm_title", normalize_title_col(F.col("title")))
    .withColumn("join_key", F.concat_ws("_", F.col("norm_title"), F.col("release_year")))
)

n_movies_raw = movies.count()
movies_clean = movies.filter(
    (F.col("budget") > 0) & (F.col("revenue") > 0) & (F.col("runtime") > 0)
    & (F.col("status") == "Released")
    & (F.col("release_year").between(MIN_YEAR, MAX_YEAR))
)

n_movies_clean = movies_clean.count()
print(f"[Source 1: movies.csv] {n_movies_raw:,} raw rows -> {n_movies_clean:,} after quality "
      f"filters (budget>0, revenue>0, runtime>0, Released, {MIN_YEAR}-{MAX_YEAR})")

# dataset 2: enhanced box office CSV
box_raw = spark.read.option("header", True).option("inferSchema", True).csv(BOX_OFFICE_CSV_PATH)

box = (
    box_raw
    .withColumnRenamed("Release Group", "title_box")
    .withColumnRenamed("$Worldwide", "worldwide_revenue_box")
    .withColumnRenamed("Year", "release_year")
    .withColumn("norm_title", normalize_title_col(F.col("title_box")))
    .withColumn("join_key", F.concat_ws("_", F.col("norm_title"), F.col("release_year")))
    .select("join_key", "worldwide_revenue_box")
)


box_window = Window.partitionBy("join_key").orderBy(F.lit(1)) #removes duplicate join keys

box_dedup = (
    box.withColumn("_rn", F.row_number().over(box_window))
       .filter(F.col("_rn") == 1)
       .drop("_rn")
)

print(f"[Source 2: box office CSV] {box.count():,} rows -> {box_dedup.count():,} deduplicated keys")

[Source 1: movies.csv] 9,770 raw rows -> 2,550 after quality filters (budget>0, revenue>0, runtime>0, Released, 2000-2026)
[Source 2: box office CSV] 5,000 rows -> 4,999 deduplicated keys


In [44]:
# dataset 3: TMDB_all_movies.csv
tmdb_all_movies_raw = (
    spark.read
    .option("header", True)
    .option("quote", "\"")
    .option("escape", "\"")
    .option("multiLine", True)
    .option("mode", "PERMISSIVE")
    .csv(TMDB_ALL_MOVIES_CSV_PATH)
)

print(f"Number of rows in TMDB_all_movies.csv: {tmdb_all_movies_raw.count()}")

tmdb_new_source3 = (
    tmdb_all_movies_raw
    .withColumn("budget", F.col("budget").cast("double"))
    .withColumn("revenue", F.col("revenue").cast("double"))
    .withColumn("runtime", F.col("runtime").cast("double"))
    .withColumn("vote_average", F.col("vote_average").cast("double"))
    .withColumn("vote_count", F.col("vote_count").cast("double"))
    .withColumn("popularity", F.col("popularity").cast("double"))
    .withColumn("release_date", F.to_date("release_date"))
    .withColumn("release_year", F.year("release_date"))
    .withColumn("norm_title", normalize_title_col(F.col("title")))
    .withColumn("join_key", F.concat_ws("_", F.col("norm_title"), F.col("release_year")))
    .withColumnRenamed("certification", "certification_us")
    .filter(
        (F.col("revenue") > 0) & (F.col("budget") > 0) & (F.col("runtime") > 0)
        & (F.col("status") == "Released")
        & (F.col("vote_average").isNotNull())
        & (F.col("production_countries").isNotNull())
        & (F.col("certification_us").isNotNull())
        & (F.col("release_year").between(MIN_YEAR, MAX_YEAR))
    )
    .withColumnRenamed("vote_average", "critic_score")
    .withColumnRenamed("revenue", "worldwide_revenue_tmdb")
    .select("join_key", "critic_score", "worldwide_revenue_tmdb", "production_countries", "certification_us")
    .drop_duplicates(subset=["join_key"])
)

n_tmdb_source3_raw = tmdb_all_movies_raw.count()
n_tmdb_source3_clean = tmdb_new_source3.count()
print(f"[Source 3: TMDB_all_movies.csv] {n_tmdb_source3_raw:,} raw rows -> {n_tmdb_source3_clean:,} usable joined rows (with production_countries and certification_us)")

Number of rows in TMDB_all_movies.csv: 1237112
[Source 3: TMDB_all_movies.csv] 1,237,112 raw rows -> 5,896 usable joined rows (with production_countries and certification_us)


## Feature Engineering

In [45]:
master_df = (
    movies_clean
    .join(box_dedup, on="join_key", how="left")
    .join(tmdb_new_source3, on="join_key", how="left")
) #create the Master DataFrame

n_matched_box = master_df.filter(F.col("worldwide_revenue_box").isNotNull()).count()
n_matched_sqlite = master_df.filter(F.col("critic_score").isNotNull()).count()

print(f"[join] Base rows: {n_movies_clean:,} | matched with box office: {n_matched_box:,} "
      f"({n_matched_box / n_movies_clean:.1%}) | matched with new dataset 3 (TMDB_all_movies.csv critic_score): {n_matched_sqlite:,} "
      f"({n_matched_sqlite / n_movies_clean:.1%})")


# revenue reconciliation
revenue_cols = ["revenue", "worldwide_revenue_box", "worldwide_revenue_tmdb"]
rev_arr = "array(" + ",".join(revenue_cols) + ")" #creates an array of value
non_null_rev = f"filter({rev_arr}, x -> x is not null)" #removes missing value

master_df = master_df.withColumn(
    "worldwide_revenue_reconciled",
    F.expr(f"aggregate({non_null_rev}, cast(0.0 as double), (acc, x) -> acc + x) "
           f"/ size({non_null_rev})")
) #calculates reconciled revenue

master_df = master_df.withColumn(
    "revenue_source_count",
    F.aggregate(
        F.array(*[F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in revenue_cols]),
        F.lit(0), lambda acc, x: acc + x,
    ),
)

# critic_score
critic_median = master_df.approxQuantile("critic_score", [0.5], 0.01)  #calculates the median
critic_median_val = critic_median[0] if critic_median and critic_median[0] is not None else 50.0

master_df = master_df.withColumn(
    "critic_score_imputed", F.coalesce(F.col("critic_score"), F.lit(critic_median_val))
) # impute missing critic scores

n_critic_missing = master_df.filter(F.col("critic_score").isNull()).count()

print(f"[cleaning] Imputed {n_critic_missing:,} missing critic_score values "
      f"with median = {critic_median_val:.1f}")

# feature engineering
for genre in GENRE_LIST:
    col_name = f"genre_{genre.lower().replace(' ', '_')}"
    master_df = master_df.withColumn(col_name, F.col("genres").contains(genre).cast("int")) # create binary genre features

master_df = (
    master_df
    .withColumn("log_budget", F.log1p("budget"))
    .withColumn("log_revenue", F.log1p("worldwide_revenue_reconciled"))
    .withColumn("is_summer_release", F.col("release_month").isin([5, 6, 7, 8]).cast("int"))
    .withColumn("is_holiday_release", F.col("release_month").isin([11, 12]).cast("int"))
    .withColumn("release_month_sin", F.sin(2 * np.pi * F.col("release_month") / 12)) # converts the month into a cyclical numerical representation
    .withColumn("release_month_cos", F.cos(2 * np.pi * F.col("release_month") / 12))
)

master_df = master_df.filter(F.col("worldwide_revenue_reconciled") > 0)
master_df.cache()

# train/test split
train_sdf = master_df.filter(F.col("release_year") <= TRAIN_MAX_YEAR)
test_sdf = master_df.filter(F.col("release_year") > TRAIN_MAX_YEAR)
n_train, n_test = train_sdf.count(), test_sdf.count()
print(f"[split] Train (<=\u00a0{TRAIN_MAX_YEAR}): {n_train:,} | Test (> {TRAIN_MAX_YEAR}): {n_test:,}")

master_df = master_df.drop("adult", "video", "created_at", "updated_at", "poster_path", "backdrop_path","homepage")

print("Master DataFrame Schema:")
master_df.printSchema()

print("\nFirst 5 rows of Master DataFrame:")
master_df.show(5, truncate=False)

print(f"\nTotal rows in Master DataFrame: {master_df.count():,}")

[join] Base rows: 2,550 | matched with box office: 1,795 (70.4%) | matched with new dataset 3 (TMDB_all_movies.csv critic_score): 2,283 (89.5%)
[cleaning] Imputed 267 missing critic_score values with median = 6.7
[split] Train (<= 2019): 1,897 | Test (> 2019): 653
Master DataFrame Schema:
root
 |-- join_key: string (nullable = false)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- runtime: double (nullable = true)
 |-- budget: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- vote_average: double (nullable = true)
 |-- vote_count: double (nullable = true)
 |-- popularity: double (nullable = true)
 |-- status: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release

In [9]:
master_pdf = master_df.toPandas()
output_path = "master_df.csv"
master_pdf.to_csv(output_path, index=False)
print(f"Master DataFrame saved to {output_path}")

Master DataFrame saved to master_df.csv


## Model Building

In [46]:
genre_feature_cols = [f"genre_{g.lower().replace(' ', '_')}" for g in GENRE_LIST]
numeric_feature_cols = [
    "log_budget", "runtime", "vote_average", "vote_count", "popularity",
    "critic_score_imputed", "is_summer_release", "is_holiday_release",
    "release_month_sin", "release_month_cos",
] + genre_feature_cols

# combining features into one vector
spark_assembler = VectorAssembler(inputCols=numeric_feature_cols, outputCol="features", handleInvalid="skip")

In [50]:
# Random Forest(PySpark MLlib)
spark_rf = SparkRFRegressor(featuresCol="features", labelCol="log_revenue", seed=RANDOM_SEED) #initialize the model

rf_pipeline = Pipeline(stages=[spark_assembler, spark_rf]) #create pipeline

param_grid_rf = (ParamGridBuilder()
              .addGrid(spark_rf.numTrees, [20, 50])
              .addGrid(spark_rf.maxDepth, [3, 6])
              .build()) #creating the RF hyperparameter grid

evaluator_rmse = RegressionEvaluator(labelCol="log_revenue", predictionCol="prediction", metricName="rmse") #creating the RMSE evaluator

cv_rf = CrossValidator(estimator=rf_pipeline, estimatorParamMaps=param_grid_rf,
                     evaluator=evaluator_rmse, numFolds=3, seed=RANDOM_SEED, parallelism=2) #creating 3 fold crossvalidation

cv_model_rf = cv_rf.fit(train_sdf) #training the cross-validation model
spark_rf_model = cv_model_rf.bestModel #getting the best model (configuration)
best_rf_stage = spark_rf_model.stages[-1] #gets the trained model

best_rf_num_trees = best_rf_stage.getOrDefault(best_rf_stage.numTrees)
best_rf_max_depth = best_rf_stage.getOrDefault(best_rf_stage.maxDepth)

print(f"[Spark RF] Best hyperparameters: numTrees={best_rf_num_trees}, "f"maxDepth={best_rf_max_depth}")

rf_test_predictions = spark_rf_model.transform(test_sdf) #making predictions on the test set
rf_spark_rmse = evaluator_rmse.evaluate(rf_test_predictions) #calculating RMSE

rf_pred_pdf = rf_test_predictions.select("log_revenue", "prediction").toPandas() #converting predictions to Pandas
rf_preds = rf_pred_pdf["prediction"].values #extracting prediction column as a NumPy array

print("Random Forest(PySpark MLlib) build complete")

[Spark RF] Best hyperparameters: numTrees=50, maxDepth=6
Random Forest(PySpark MLlib) build complete


In [12]:
# Linear Regression(PySpark MLlib)
spark_lr = SparkLinearRegression(featuresCol="features", labelCol="log_revenue") #initialize the model

lr_pipeline = Pipeline(stages=[spark_assembler, spark_lr]) #create pipeline

spark_lr_model = lr_pipeline.fit(train_sdf) #training the model

lr_test_predictions = spark_lr_model.transform(test_sdf) #making predictions on the test set
lr_spark_rmse = evaluator_rmse.evaluate(lr_test_predictions) #calculating RMSE

lr_pred_pdf = lr_test_predictions.select("log_revenue", "prediction").toPandas() #converting predictions to Pandas
lr_preds = lr_pred_pdf["prediction"].values #extracting prediction column as a NumPy array

print("Linear Regression(PySpark MLlib) build complete")

Linear Regression(PySpark MLlib) build complete


In [13]:
# GBTRegressor(PySpark MLlib)
spark_gbt = SparkGBTRegressor(featuresCol="features", labelCol="log_revenue", seed=RANDOM_SEED) #initialize the model

gbt_pipeline = Pipeline(stages=[spark_assembler, spark_gbt]) #create pipeline

param_grid_gbt = (ParamGridBuilder()
              .addGrid(spark_gbt.maxIter, [10, 20])
              .addGrid(spark_gbt.maxDepth, [3, 5])
              .build()) #hyperparameter grid

cv_gbt = CrossValidator(estimator=gbt_pipeline, estimatorParamMaps=param_grid_gbt,
                     evaluator=evaluator_rmse, numFolds=3, seed=RANDOM_SEED, parallelism=2)

cv_model_gbt = cv_gbt.fit(train_sdf)  #training the cross cvalidation model
spark_gbt_model = cv_model_gbt.bestModel #getting the best model (configuration)
best_gbt_stage = spark_gbt_model.stages[-1] #gets the trained model

print(f"[Spark GBT] Best hyperparameters: maxIter={best_gbt_stage.getMaxIter()}, "
      f"maxDepth={best_gbt_stage.getMaxDepth()}")

best_gbt_max_iter = best_gbt_stage.getMaxIter()
best_gbt_max_depth = best_gbt_stage.getMaxDepth()

gbt_test_predictions = spark_gbt_model.transform(test_sdf) #making predictions on the test set
gbt_spark_rmse = evaluator_rmse.evaluate(gbt_test_predictions) #calculating RMSE

gbt_pred_pdf = gbt_test_predictions.select("log_revenue", "prediction").toPandas() #converting predictions to Pandas
gbt_preds = gbt_pred_pdf["prediction"].values #extracting prediction column as a NumPy array

print("GBTRegressor(PySpark MLlib) build complete")

[Spark GBT] Best hyperparameters: maxIter=20, maxDepth=3
GBTRegressor(PySpark MLlib) build complete


In [14]:
# SparkXGBRegressor(PySpark MLlib)
spark_xgb = SparkXGBRegressor(features_col="features", label_col="log_revenue",
                            n_estimators=300, max_depth=5, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=RANDOM_SEED) #initialize the model

xgb_pipeline = Pipeline(stages=[spark_assembler, spark_xgb]) #create pipelie

spark_xgb_model = xgb_pipeline.fit(train_sdf) #training the model

xgb_test_predictions = spark_xgb_model.transform(test_sdf) #making predictions on the test set
xgb_spark_rmse = evaluator_rmse.evaluate(xgb_test_predictions) #calculating RMSE

xgb_pred_pdf = xgb_test_predictions.select("log_revenue", "prediction").toPandas() #converting predictions to Pandas
xgb_preds = xgb_pred_pdf["prediction"].values #extracting prediction column as a NumPy array

print("SparkXGBRegressor(PySpark MLlib) build complete")

INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


SparkXGBRegressor(PySpark MLlib) build complete


In [15]:
# Artifical Neural Network (ANN)
pandas_master = master_df.toPandas() #converts spark DataFrame to Pandas
target_col = "log_revenue"
feature_cols = numeric_feature_cols #define the input features

train_mask = pandas_master["release_year"] <= TRAIN_MAX_YEAR
test_mask = pandas_master["release_year"] > TRAIN_MAX_YEAR

X_train = pandas_master.loc[train_mask, feature_cols].reset_index(drop=True) # contains the input features for the training data
y_train = pandas_master.loc[train_mask, target_col].reset_index(drop=True)

X_test = pandas_master.loc[test_mask, feature_cols].reset_index(drop=True) # contains the input features for the testing data
y_test = pandas_master.loc[test_mask, target_col].reset_index(drop=True)

for col in X_train.columns: # median imputation/handling missing values
    if X_train[col].isnull().any():
        median_val = X_train[col].median() #calcaulate median value
        X_train[col] = X_train[col].fillna(median_val) #fill median value in training data
        X_test[col] = X_test[col].fillna(median_val) #fill median value in testing data

ann_scaler = StandardScaler().fit(X_train) #standardize the features

X_train_scaled = ann_scaler.transform(X_train) #transform the training features
X_test_scaled = ann_scaler.transform(X_test) #transform the test features

ann_model = MLPRegressor(hidden_layer_sizes=(128, 64), activation='relu', solver='adam',
                          max_iter=1000, early_stopping=True, n_iter_no_change=20,
                          random_state=RANDOM_SEED) #initialize the model

ann_model.fit(X_train_scaled, y_train) #training the model
ann_preds = ann_model.predict(X_test_scaled) #making predictions on the test set

print("ANN build complete")

ANN build complete


## Data Visualization

In [16]:
plt.figure()
sns.histplot(pandas_master['worldwide_revenue_reconciled'] / 1e6, kde=True, color='blue')
plt.title("Distribution of Worldwide Revenue (Millions USD)")
plt.xlabel("Revenue ($ Millions)")
plt.ylabel("Frequency")
plt.savefig("visualizations/revenue_distribution.png")
plt.close()

plt.figure()
pandas_master['budget_group'] = pd.qcut(pandas_master['budget'], q=5,labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
pandas_master['worldwide_revenue_millions'] = pandas_master['worldwide_revenue_reconciled'] / 1e6
sns.boxplot(data=pandas_master, x='budget_group', y='worldwide_revenue_millions',
            hue='budget_group', palette="Blues", legend=False)
plt.title("Worldwide Box Office Revenue by Budget Group")
plt.xlabel("Budget Tier")
plt.ylabel("Revenue ($ Millions)")
plt.savefig("visualizations/budget_vs_revenue_boxplot.png")
plt.close()

plt.figure(figsize=(10, 8))
corr_matrix = pandas_master[feature_cols + [target_col]].corr()
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, cbar=True)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("visualizations/correlation_heatmap.png")
plt.close()

plt.figure()
yearly_trend = pandas_master.groupby("release_year")["worldwide_revenue_reconciled"].mean().reset_index()
yearly_trend['worldwide_revenue_millions'] = yearly_trend['worldwide_revenue_reconciled'] / 1e6
sns.lineplot(data=yearly_trend, x="release_year", y='worldwide_revenue_millions', marker="o", color="green")
plt.title("Average Yearly Worldwide Box Office Revenue Trends (Real Data)")
plt.xlabel("Year")
plt.ylabel("Average Revenue ($ Millions)")
plt.savefig("visualizations/yearly_revenue_trends.png")
plt.close()

plt.figure(figsize=(10, 5))
genre_counts = pandas_master[genre_feature_cols].sum().sort_values(ascending=False)
genre_counts.index = [c.replace("genre_", "").replace("_", " ").title() for c in genre_counts.index]
sns.barplot(x=genre_counts.values, y=genre_counts.index, hue=genre_counts.index,
            palette="viridis", legend=False)
plt.title("Number of Films per Genre (Multi-label -- Films May Count Toward >1 Genre)")
plt.xlabel("Number of Films")
plt.ylabel("Genre")
plt.tight_layout()
plt.savefig("visualizations/genre_frequency.png")
plt.close()

print("Visualizations successfully saved in 'visualizations/' folder")

Visualizations successfully saved in 'visualizations/' folder


## Model Evaluation

In [17]:
evaluation_results = {}

def calculate_mape(y_true, y_pred, eps=1.0):
    denom = np.clip(np.abs(y_true), eps, None)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100 #calculate percentage error


def evaluate_model(model_name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = calculate_mape(np.asarray(y_true), np.asarray(y_pred))
    r2 = r2_score(y_true, y_pred)
    evaluation_results[model_name] = {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2 Score": r2}
    print(f"\n--- {model_name} Performance (log-revenue space) ---")
    print(f"RMSE: {rmse:.4f}   MAE: {mae:.4f}   MAPE: {mape:.2f}%   R Square: {r2:.4f}")

evaluate_model("Baseline Linear Regression (PySpark MLlib)", y_test.values, lr_preds)

evaluate_model("Random Forest (PySpark MLlib)", y_test.values, rf_preds)

evaluate_model("GBTRegressor (PySpark MLlib)", y_test.values, gbt_preds)

evaluate_model("SparkXGBRegressor", y_test.values, xgb_preds)

evaluate_model("Artificial Neural Network", y_test, ann_preds)


--- Baseline Linear Regression (PySpark MLlib) Performance (log-revenue space) ---
RMSE: 2.1635   MAE: 1.3568   MAPE: 9.94%   R Square: 0.1587

--- Random Forest (PySpark MLlib) Performance (log-revenue space) ---
RMSE: 1.7403   MAE: 1.1175   MAPE: 8.34%   R Square: 0.4556

--- GBTRegressor (PySpark MLlib) Performance (log-revenue space) ---
RMSE: 1.8523   MAE: 1.2280   MAPE: 8.84%   R Square: 0.3833

--- SparkXGBRegressor Performance (log-revenue space) ---
RMSE: 1.7592   MAE: 1.1560   MAPE: 8.39%   R Square: 0.4438

--- Artificial Neural Network Performance (log-revenue space) ---
RMSE: 8.0980   MAE: 2.2268   MAPE: 14.49%   R Square: -10.7871


## Hybrid Model Build and Evaluation

In [18]:
def fit_predict_spark_rf_fold(X_tr_fold, y_tr_fold, X_ho_fold, num_trees, max_depth): # trains a RF model on one training fold
    fold_train_pdf = X_tr_fold.copy() # copying the training features
    fold_train_pdf["log_revenue"] = y_tr_fold.values # adding the target column
    # converting Pandas to Spark
    fold_train_sdf = spark.createDataFrame(fold_train_pdf)
    fold_ho_sdf = spark.createDataFrame(X_ho_fold)

    # creating the fold feature assembler
    fold_assembler = VectorAssembler(inputCols=numeric_feature_cols, outputCol="features",
                                      handleInvalid="skip")

    fold_rf = SparkRFRegressor(featuresCol="features", labelCol="log_revenue",
                                numTrees=num_trees, maxDepth=max_depth, seed=RANDOM_SEED) #initilaize the model
    fold_pipeline = Pipeline(stages=[fold_assembler, fold_rf]) #create pipeline
    fold_model = fold_pipeline.fit(fold_train_sdf) #training the fold model

    fold_preds_pdf = fold_model.transform(fold_ho_sdf).select("prediction").toPandas() #generating fold predictions
    return fold_preds_pdf["prediction"].values

def fit_predict_spark_xgb_fold(X_tr_fold, y_tr_fold, X_ho_fold,
                               n_estimators, max_depth, learning_rate,
                               subsample, colsample_bytree): # trains a Spark XGB model on one training fold
    fold_train_pdf = X_tr_fold.copy() # copying the training features
    fold_train_pdf["log_revenue"] = y_tr_fold.values # adding the target column
    # converting Pandas to Spark
    fold_train_sdf = spark.createDataFrame(fold_train_pdf)
    fold_ho_sdf = spark.createDataFrame(X_ho_fold)

    # creating the fold feature assembler
    fold_assembler = VectorAssembler(inputCols=numeric_feature_cols, outputCol="features",
                                      handleInvalid="skip")

    fold_xgb = SparkXGBRegressor(features_col="features", label_col="log_revenue",
                                 n_estimators=n_estimators, max_depth=max_depth,
                                 learning_rate=learning_rate, subsample=subsample,
                                 colsample_bytree=colsample_bytree,
                                 random_state=RANDOM_SEED) #initilaize the model

    fold_pipeline = Pipeline(stages=[fold_assembler, fold_xgb]) #create pipeline
    fold_model = fold_pipeline.fit(fold_train_sdf) #training the fold model

    fold_preds_pdf = fold_model.transform(fold_ho_sdf).select("prediction").toPandas() #generating fold predictions
    return fold_preds_pdf["prediction"].values


print("Building Hybrid stacking ensemble (Random Forest + SparkXGBRegressor via Ridge meta-learner)")
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED) #creating 5 fold cross-validation
oof_preds = pd.DataFrame(index=X_train.index, columns=["rf", "xgb"], dtype=float) #creating the out-of-fold prediction DataFrame

# 5 fold loop
for fold_idx, (tr_idx, ho_idx) in enumerate(kf.split(X_train)):
    # extracting training and holdout data
    X_tr, X_ho = X_train.iloc[tr_idx], X_train.iloc[ho_idx]
    y_tr = y_train.iloc[tr_idx]

    # generate RF OOF predictions
    oof_preds.iloc[ho_idx, oof_preds.columns.get_loc("rf")] = fit_predict_spark_rf_fold(
        X_tr, y_tr, X_ho, best_rf_num_trees, best_rf_max_depth
    )

    # generate SparkXGBRegressor OOF predictions
    oof_preds.iloc[ho_idx, oof_preds.columns.get_loc("xgb")] = fit_predict_spark_xgb_fold(
        X_tr, y_tr, X_ho,
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8
    )

    print(f"Hybrid stacking fold {fold_idx + 1}/5 complete (RF + SparkXGBRegressor)")

meta_learner = Ridge(alpha=1.0, random_state=RANDOM_SEED) # initialize the Ridge meta learner
meta_learner.fit(oof_preds, y_train) #training the model

meta_model = meta_learner #assigning the meta model

hybrid_test_features = pd.DataFrame({"rf": rf_preds, "xgb": xgb_preds}) #create the hybrid test features
hybrid_preds = meta_learner.predict(hybrid_test_features) #generating final hybrid predictions

evaluate_model("Hybrid (Ridge-stacked RF + SparkXGBRegressor)", y_test, hybrid_preds)

print(f"Meta-learner weights -> RF: {meta_learner.coef_[0]:.3f}, "
      f"XGBoost: {meta_learner.coef_[1]:.3f}, "
      f"intercept: {meta_learner.intercept_:.3f}")

Building Hybrid stacking ensemble (Random Forest + SparkXGBRegressor via Ridge meta-learner)


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Hybrid stacking fold 1/5 complete (RF + SparkXGBRegressor)


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Hybrid stacking fold 2/5 complete (RF + SparkXGBRegressor)


...

Hybrid stacking fold 3/5 complete (RF + SparkXGBRegressor)


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Hybrid stacking fold 4/5 complete (RF + SparkXGBRegressor)


INFO:XGBoost-PySpark:Running xgboost-3.4.1 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 300}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Hybrid stacking fold 5/5 complete (RF + SparkXGBRegressor)

--- Hybrid (Ridge-stacked RF + SparkXGBRegressor) Performance (log-revenue space) ---
RMSE: 1.7257   MAE: 1.1213   MAPE: 8.22%   R Square: 0.4648
Meta-learner weights -> RF: 0.481, XGBoost: 0.563, intercept: -0.797


In [19]:
summary_df = pd.DataFrame(evaluation_results).T.sort_values("RMSE")
print(f"\nModel Evaluation Summary Table:")
print(summary_df)

best_model_name = summary_df.index[0]
print(f"\nBest model by test RMSE: {best_model_name}")


Model Evaluation Summary Table:
                                                   RMSE       MAE   MAPE (%)  \
Hybrid (Ridge-stacked RF + SparkXGBRegressor)  1.725652  1.121307   8.223296   
Random Forest (PySpark MLlib)                  1.740333  1.117467   8.340928   
SparkXGBRegressor                              1.759179  1.156004   8.389520   
GBTRegressor (PySpark MLlib)                   1.852314  1.228016   8.841781   
Baseline Linear Regression (PySpark MLlib)     2.163478  1.356761   9.936883   
Artificial Neural Network                      8.098049  2.226781  14.485764   

                                                R2 Score  
Hybrid (Ridge-stacked RF + SparkXGBRegressor)   0.464754  
Random Forest (PySpark MLlib)                   0.455608  
SparkXGBRegressor                               0.443754  
GBTRegressor (PySpark MLlib)                    0.383297  
Baseline Linear Regression (PySpark MLlib)      0.158698  
Artificial Neural Network                     -10.78

## Model Comparison Visualization

In [20]:
plt.figure(figsize=(9, 5))
plot_df = summary_df.reset_index().rename(columns={"index": "model"})
sns.barplot(data=plot_df, x="RMSE", y="model", hue="model", palette="viridis", legend=False)
plt.title("Model Comparison: Test-Set RMSE (log-revenue space)")
plt.xlabel("RMSE (lower is better)")
plt.ylabel("")
plt.tight_layout()
plt.savefig("visualizations/model_comparison_rmse.png")
plt.close()


best_model_preds = {
    "Baseline Linear Regression (PySpark MLlib)": lr_preds,
    "Random Forest (PySpark MLlib)": rf_preds,
    "GBTRegressor (PySpark MLlib)": gbt_preds,
    "SparkXGBRegressor": xgb_preds,
    "Artificial Neural Network": ann_preds,
    "Hybrid (Ridge-stacked RF + SparkXGBRegressor)": hybrid_preds,
}[best_model_name]

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_model_preds, alpha=0.4, edgecolor="none")
lims = [min(y_test.min(), best_model_preds.min()), max(y_test.max(), best_model_preds.max())]
plt.plot(lims, lims, color="red", linestyle="--", label="Perfect prediction")
plt.xlabel("Actual log(1 + revenue)")
plt.ylabel("Predicted log(1 + revenue)")
plt.title(f"Predicted vs. Actual -- {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig("visualizations/predicted_vs_actual.png")
plt.close()

plt.figure(figsize=(9, 5))
residuals = y_test.values - best_model_preds
plt.scatter(best_model_preds, residuals, alpha=0.4, edgecolor="none")
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted log(1 + revenue)")
plt.ylabel("Residual (actual - predicted)")
plt.title(f"Residual Plot -- {best_model_name}")
plt.tight_layout()
plt.savefig("visualizations/residual_plot.png")
plt.close()

base_residuals = pd.DataFrame({
    "RF": y_test.values - rf_preds,
    "XGBoost": y_test.values - xgb_preds,
})
residual_corr = base_residuals.corr()

plt.figure(figsize=(6, 5))
sns.heatmap(residual_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation Between Base Models' Prediction Errors\n(lower = more diverse = better stacking candidates)")
plt.tight_layout()
plt.savefig("visualizations/base_model_error_correlation.png")
plt.close()
print("\n[diagnostics] Base-model residual correlation matrix:")
print(residual_corr)


[diagnostics] Base-model residual correlation matrix:
               RF   XGBoost
RF       1.000000  0.937761
XGBoost  0.937761  1.000000


# XAI

In [21]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from lime.lime_tabular import LimeTabularExplainer
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

XAI_DIR = "visualizations/xai"
os.makedirs(XAI_DIR, exist_ok=True)

INSTANCE = 7

# Data Preparation converts to Pandas
train_pdf = train_sdf.select(feature_cols + ["log_revenue"]).toPandas()
test_pdf  = test_sdf.select(feature_cols + ["log_revenue"]).toPandas()

X_train = train_pdf[feature_cols]
y_train = train_pdf["log_revenue"]

X_test = test_pdf[feature_cols]
y_test = test_pdf["log_revenue"]

medians = X_train.median().to_dict() #calculates the median of every training feature

print("STARTING XAI ANALYSIS")
print(f"Training: {X_train.shape}")
print(f"Testing : {X_test.shape}")

STARTING XAI ANALYSIS
Training: (1897, 22)
Testing : (653, 22)


## SHAP

In [22]:
def run_shap(model, name):

    model.fit(X_train, y_train)

    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X_test)

    # Global importance
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Mean_Absolute_SHAP": np.abs(values).mean(axis=0)
    }).sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    importance["Rank"] = np.arange(1, len(importance) + 1)

    # Beeswarm
    shap.summary_plot(
        values,
        X_test,
        max_display=15,
        show=False
    )

    plt.title(f"SHAP Summary Plot - {name}")
    plt.tight_layout()
    plt.savefig(
        f"{XAI_DIR}/SHAP_{name}_Beeswarm.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # Waterfall
    base = explainer.expected_value
    if isinstance(base, np.ndarray):
        base = base.item()

    explanation = shap.Explanation(
        values=values[INSTANCE],
        base_values=base,
        data=X_test.iloc[INSTANCE].values,
        feature_names=feature_cols
    )

    shap.plots.waterfall(
        explanation,
        max_display=10,
        show=False
    )

    plt.tight_layout()
    plt.savefig(
        f"{XAI_DIR}/SHAP_{name}_Waterfall_Instance_{INSTANCE}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(f"\n{name} SHAP Top 10:")
    print(importance.head(10).to_string(index=False))

    return model, values, importance

In [23]:
# SHAP - XGBRegressor
print("SHAP - XGBRegressor")

xgb_proxy = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=1
)

xgb_proxy, xgb_shap, xgb_importance = run_shap(xgb_proxy, "XGB")

SHAP - XGBRegressor


/tmp/ipykernel_5532/1736082489.py:20: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(



XGB SHAP Top 10:
              Feature  Mean_Absolute_SHAP  Rank
           vote_count            0.974683     1
           log_budget            0.750085     2
           popularity            0.126292     3
         vote_average            0.117484     4
              runtime            0.106982     5
          genre_drama            0.092832     6
genre_science_fiction            0.075078     7
         genre_comedy            0.068048     8
 critic_score_imputed            0.058296     9
    release_month_cos            0.046119    10


In [24]:
# SHAP - Random Forest
print("SHAP - Random Forest")

RF_NUM_TREES = best_rf_num_trees
RF_MAX_DEPTH = best_rf_max_depth

rf_proxy = RandomForestRegressor(
    n_estimators=RF_NUM_TREES,
    max_depth=RF_MAX_DEPTH,
    random_state=42,
    n_jobs=1
)

rf_proxy, rf_shap, rf_importance = run_shap(
    rf_proxy,
    "RF"
)

SHAP - Random Forest


/tmp/ipykernel_5532/1736082489.py:20: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(



RF SHAP Top 10:
              Feature  Mean_Absolute_SHAP  Rank
           vote_count            1.227722     1
           log_budget            0.779741     2
         vote_average            0.074440     3
           popularity            0.043508     4
         genre_family            0.025637     5
              runtime            0.024554     6
         genre_comedy            0.023738     7
 critic_score_imputed            0.018830     8
genre_science_fiction            0.018478     9
    release_month_cos            0.013314    10


## SHAP PySpark Model

In [25]:
def run_shap_kernel(spark_model, feature_cols, name, background_size=50, test_sample_size=50):

    def spark_predict(X_numpy):
        pdf = pd.DataFrame(X_numpy, columns=feature_cols) #convert NumPy input into Pandas
        sdf = spark.createDataFrame(pdf) #convert Pandas into Spark
        preds = spark_model.transform(sdf).select("prediction").toPandas() #generate predictions using the native Spark model
        return preds["prediction"].values

    background = shap.sample(X_train, background_size, random_state=RANDOM_SEED) #selecting the SHAP background dataset

    X_test_sample = X_test.sample(test_sample_size, random_state=RANDOM_SEED).reset_index(drop=True) #sampling the test dataset

    explainer = shap.KernelExplainer(spark_predict, background) #initializing the KernelSHAP explainer

    values = explainer.shap_values(X_test_sample) #calculating KernelSHAP values

    # Global importance
    importance = pd.DataFrame({
        "Feature": feature_cols,
        "Mean_Absolute_SHAP": np.abs(values).mean(axis=0)
    }).sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    importance["Rank"] = np.arange(1, len(importance) + 1)

    # Beeswarm
    shap.summary_plot(
        values,
        X_test_sample,
        max_display=15,
        show=False
    )

    plt.title(f"SHAP Summary Plot - {name} (KernelExplainer, PySpark model)")
    plt.tight_layout()
    plt.savefig(
        f"{XAI_DIR}/SHAP_{name}_Beeswarm_Kernel.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    # Waterfall
    base = explainer.expected_value #retrieves the expected model prediction

    if isinstance(base, np.ndarray):
        base = base.item() #converts to a scaler

    local_instance = min(INSTANCE, len(X_test_sample) - 1) #selecting the local explanation instance

    explanation = shap.Explanation(
        values=values[local_instance],
        base_values=base,
        data=X_test_sample.iloc[local_instance].values,
        feature_names=feature_cols
    ) #initializing the local SHAP explanation

    shap.plots.waterfall(
        explanation,
        max_display=10,
        show=False
    )

    plt.tight_layout()
    plt.savefig(
        f"{XAI_DIR}/SHAP_{name}_Waterfall_Instance_{local_instance}_Kernel.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    print(f"\n{name} SHAP Top 10 (KernelExplainer on native PySpark model):")
    print(importance.head(10).to_string(index=False))

    return spark_model, values, importance

In [26]:
# SHAP - Random Forest (native PySpark MLlib model)
print("SHAP - Random Forest (PySpark MLlib, KernelExplainer)")

rf_shap_model, rf_shap_kernel, rf_importance_kernel = run_shap_kernel(
    spark_rf_model,
    feature_cols,
    "RF"
)

SHAP - Random Forest (PySpark MLlib, KernelExplainer)


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_5532/79605946.py:29: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(



RF SHAP Top 10 (KernelExplainer on native PySpark model):
             Feature  Mean_Absolute_SHAP  Rank
          vote_count            0.651314     1
          log_budget            0.607307     2
          popularity            0.165063     3
     genre_adventure            0.053677     4
             runtime            0.047446     5
        vote_average            0.042865     6
critic_score_imputed            0.031994     7
   release_month_sin            0.020214     8
         genre_drama            0.019298     9
        genre_family            0.014762    10


In [27]:
# SHAP - XGBRegressor (native PySpark model)
print("SHAP - XGBRegressor (PySpark, KernelExplainer)")

xgb_shap_model, xgb_shap_kernel, xgb_importance_kernel = run_shap_kernel(
    spark_xgb_model,
    feature_cols,
    "XGB"
)

SHAP - XGBRegressor (PySpark, KernelExplainer)


  0%|          | 0/50 [00:00<?, ?it/s]

/tmp/ipykernel_5532/79605946.py:29: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(



XGB SHAP Top 10 (KernelExplainer on native PySpark model):
              Feature  Mean_Absolute_SHAP  Rank
           vote_count            0.779946     1
           log_budget            0.691348     2
           popularity            0.103059     3
          genre_drama            0.098676     4
         vote_average            0.094088     5
genre_science_fiction            0.094045     6
              runtime            0.069666     7
         genre_comedy            0.044573     8
 critic_score_imputed            0.031660     9
    release_month_sin            0.030093    10


## LIME

In [28]:
# LIME Setup
categorical = [
    i for i, col in enumerate(feature_cols)
    if X_train[col].nunique() <= 2
] #creates a list containing the column indexes of categorical or binary features

categorical_names = {
    i: [str(v) for v in sorted(X_train.iloc[:, i].dropna().unique())]
    for i in categorical
} #creates a dictionary containing the possible category values for each categorical feature

lime_explainer = LimeTabularExplainer(
    X_train.values,
    feature_names=feature_cols,
    categorical_features=categorical,
    categorical_names=categorical_names,
    mode="regression",
    discretize_continuous=True,
    random_state=42
) #initializing the LIME explainer

instance = X_test.iloc[INSTANCE].values #selects test observation for local explanation

In [29]:
def spark_predict(model, X):

    if isinstance(X, np.ndarray): #check whether the input is a NumPy array
        X = pd.DataFrame(X, columns=feature_cols) #convert NumPy data into Pandas

    X = X.fillna(medians) #replaces missing values in X using the pre-calculated feature medians

    sdf = spark.createDataFrame(X) #convert Pandas into Spark

    return (
        model.transform(sdf) #apply the PySpark model
        .select("prediction") #select only the prediction column
        .toPandas()["prediction"] #convert the predictions to Pandas
        .values #extract the prediction values as a NumPy array
    )


def lime_explain(predict_fn, name):
    explanation = lime_explainer.explain_instance(
        instance,
        predict_fn,
        num_features=10,
        num_samples=3000
    ) #initializing the LIME explainer

    results = explanation.as_list() #extract the LIME results

    df = pd.DataFrame(
        results,
        columns=["Feature_Condition", "LIME_Weight"]
    ) #create a results DataFrame

    df["Rank"] = np.arange(1, len(df) + 1) #create a Rank DataFrame

    df["Direction"] = np.where(
        df["LIME_Weight"] > 0,
        "Positive",
        "Negative"
    ) #create a Direction column

    # Plot
    plt.figure(figsize=(10, 6))

    plt.barh(
        np.arange(len(results)),
        [x[1] for x in results]
    )

    plt.yticks(
        np.arange(len(results)),
        [x[0] for x in results]
    )

    plt.xlabel("LIME Contribution Weight")
    plt.title(f"LIME Local Explanation - {name}")
    plt.axvline(0, linewidth=1)
    plt.gca().invert_yaxis()
    plt.tight_layout()

    plt.savefig(
        f"{XAI_DIR}/LIME_{name}_Instance_{INSTANCE}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(f"\n{name} LIME:")
    for i, (feature, weight) in enumerate(results, 1):
        print(f"{i:02d}. {feature}: {weight:.6f}")

    return results

In [30]:
# LIME - XGBRegressor
print("LIME - XGBRegressor")

xgb_lime = lime_explain(
    lambda X: spark_predict(spark_xgb_model, X),
    "XGB"
)

LIME - XGBRegressor

XGB LIME:
01. 17.37 < log_budget <= 18.13: 0.891999
02. genre_science_fiction=0.0: 0.509856
03. 1688.00 < vote_count <= 3523.00: -0.410455
04. genre_drama=0.0: 0.327291
05. genre_animation=0.0: -0.269893
06. genre_comedy=0.0: -0.256279
07. popularity > 8.75: 0.253700
08. genre_fantasy=1.0: -0.165395
09. 6.70 < critic_score_imputed <= 7.14: -0.141079
10. 6.65 < vote_average <= 7.20: 0.118597


In [31]:
# LIME - Random Forest
print("LIME - Random Forest")

rf_lime = lime_explain(
    lambda X: spark_predict(spark_rf_model, X),
    "RF"
)

LIME - Random Forest

RF LIME:
01. 17.37 < log_budget <= 18.13: 0.548120
02. popularity > 8.75: 0.322947
03. genre_adventure=0.0: -0.137790
04. genre_science_fiction=0.0: 0.124950
05. genre_horror=1.0: -0.118575
06. 1688.00 < vote_count <= 3523.00: -0.106497
07. is_summer_release=0: 0.103815
08. genre_drama=0.0: 0.098655
09. genre_family=0.0: -0.076450
10. runtime > 122.00: 0.068438


In [32]:
# Hybrid Model XAI
def hybrid_predict(X):

    rf_pred = spark_predict(spark_rf_model, X) #generate RF predictions
    xgb_pred = spark_predict(spark_xgb_model, X) #generate XGBoost predictions

    return meta_model.predict(
        np.column_stack([rf_pred, xgb_pred]) #combine the base model predictions
    )


# Hybrid Model XAI LIME
print("Hybrid Model XAI LIME")

hybrid_lime = lime_explain(
    hybrid_predict,
    "Hybrid"
)

Hybrid Model XAI LIME


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(



Hybrid LIME:
01. 17.37 < log_budget <= 18.13: 0.638999
02. genre_science_fiction=0.0: 0.220680
03. 1688.00 < vote_count <= 3523.00: -0.213426
04. popularity > 8.75: 0.199044
05. genre_family=0.0: 0.182895
06. genre_drama=0.0: 0.165220
07. genre_fantasy=1.0: -0.149545
08. genre_romance=0.0: -0.148793
09. genre_comedy=0.0: -0.139410
10. 6.65 < vote_average <= 7.20: 0.118023


In [33]:
# Hybrid Component Contributions
rf_pred = spark_predict(
    spark_rf_model,
    X_test
) #get RF Predictions

xgb_pred = spark_predict(
    spark_xgb_model,
    X_test
) #get XGBoost predictions

rf_value = rf_pred[INSTANCE] #select the prediction
xgb_value = xgb_pred[INSTANCE]

rf_weight = meta_model.coef_[0] #extract the ridge meta learner weights
xgb_weight = meta_model.coef_[1]
intercept = meta_model.intercept_ #extract the ridge intercept

rf_contribution = rf_weight * rf_value #calculate RF contribution
xgb_contribution = xgb_weight * xgb_value #calculate XGBoost contribution

component_df = pd.DataFrame({
    "Component": [
        "Random Forest",
        "SparkXGBRegressor",
        "Ridge Intercept"
    ],
    "Coefficient": [
        rf_weight,
        xgb_weight,
        1
    ],
    "Prediction": [
        rf_value,
        xgb_value,
        intercept
    ],
    "Contribution": [
        rf_contribution,
        xgb_contribution,
        intercept
    ]
})

print("Hybrid component contributions:")
print(component_df.to_string(index=False))

Hybrid component contributions:
        Component  Coefficient  Prediction  Contribution
    Random Forest     0.480774   18.509620      8.898945
SparkXGBRegressor     0.562980   18.476728     10.402037
  Ridge Intercept     1.000000   -0.796508     -0.796508


In [34]:
# Hybrid Model Component Plot
plt.figure(figsize=(8, 5))

plt.bar(
    component_df["Component"],
    component_df["Contribution"]
)

plt.ylabel("Contribution to Log-Revenue Prediction")
plt.title(
    f"Hybrid Model Component Contributions\n"
    f"Test Instance {INSTANCE}"
)

plt.axhline(0, linewidth=1)
plt.tight_layout()

plt.savefig(
    f"{XAI_DIR}/Hybrid_Component_Contributions_Instance_{INSTANCE}.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()